# Laboratorio: Curva ROC y AUC para comparar modelos de detección de fraude

## Paso 1: Crear el escenario

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score

# 2,000 transacciones, 10 características, 90 % normales / 10 % fraudes
X, y = make_classification(n_samples=2000, n_features=10,
                           weights=[0.9, 0.1], random_state=42)

# 70 % entrenamiento, 30 % prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Fraudes en test:", y_test.sum(), "de", len(y_test))

## Paso 2: Entrenar a los competidores

In [ ]:
modelo_log = LogisticRegression()
modelo_log.fit(X_train, y_train)

modelo_rf = RandomForestClassifier(random_state=42)
modelo_rf.fit(X_train, y_train)

## Paso 3: Extraer probabilidades de la clase 1 (fraude)

In [ ]:
prob_log = modelo_log.predict_proba(X_test)[:, 1]
prob_rf  = modelo_rf.predict_proba(X_test)[:, 1]

## Paso 4: Coordenadas ROC y AUC

In [ ]:
fpr_log, tpr_log, _ = roc_curve(y_test, prob_log)
auc_log = roc_auc_score(y_test, prob_log)

fpr_rf, tpr_rf, _ = roc_curve(y_test, prob_rf)
auc_rf = roc_auc_score(y_test, prob_rf)

print(f"AUC Regresión Logística: {auc_log:.3f}")
print(f"AUC Random Forest      : {auc_rf:.3f}")

## Paso 5: Graficar

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(fpr_log, tpr_log, color="blue",  label=f"Regresión Logística (AUC = {auc_log:.3f})")
plt.plot(fpr_rf,  tpr_rf,  color="green", label=f"Random Forest (AUC = {auc_rf:.3f})")
plt.plot([0, 1], [0, 1], color="red", linestyle="--", label="Adivinanza al azar (AUC = 0.500)")
plt.title("Curva ROC: Regresión Logística vs Random Forest (detección de fraude)")
plt.xlabel("Tasa de Falsos Positivos")
plt.ylabel("Tasa de Verdaderos Positivos")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Apoyo para la pregunta 2: FPR del modelo perdedor cuando TPR = 0.8

In [ ]:
def fpr_en_tpr(fpr, tpr, objetivo=0.8):
    # primer punto de la curva donde se alcanza al menos el TPR objetivo
    return fpr[np.argmax(tpr >= objetivo)]

print(f"Regresión Logística: FPR = {fpr_en_tpr(fpr_log, tpr_log):.3f}")
print(f"Random Forest      : FPR = {fpr_en_tpr(fpr_rf, tpr_rf):.3f}")

## Entregable: análisis

Resultados obtenidos con `random_state=42` (pueden variar ligeramente con otra versión de scikit-learn): AUC Regresión Logística ≈ **0.963**, AUC Random Forest ≈ **0.970**.

**1. El ganador.** Elegiría **Random Forest**: tiene el AUC más alto (≈0.970 vs 0.963), es decir, la mayor probabilidad de asignar un puntaje de riesgo más alto a un fraude real que a una transacción legítima, *en todos los umbrales posibles*. Su curva queda por encima (más cerca de la esquina superior izquierda), por lo que con el mismo nivel de falsas alarmas atrapa más fraudes. Un matiz honesto: la diferencia es pequeña (≈0.007) y con solo 71 fraudes en el conjunto de prueba puede no ser estadísticamente significativa; antes de producción convendría validación cruzada, y la regresión logística es más interpretable, algo valioso para auditoría regulatoria. Además, el AUC no depende del umbral, así que el umbral final debe elegirse por costos (FN vs FP).

**2. El riesgo comercial (modelo perdedor: Regresión Logística).** Para atrapar el 80 % de los fraudes (TPR = 0.8), la curva de la regresión logística muestra una FPR de aproximadamente **3 %** (≈0.032; el Random Forest necesitaría ≈2 %). Es decir, alrededor de 3 de cada 100 clientes honestos verían su tarjeta bloqueada por error. Con 529 transacciones legítimas en el test son ≈17 clientes; a escala de millones de transacciones, decenas de miles de bloqueos injustificados, con costo de atención al cliente y riesgo de que el cliente cambie de banco. Ambos modelos son buenos, pero ese 1 punto porcentual de diferencia se multiplica a gran escala.

**3. El peor escenario (volado de moneda).** La diagonal roja es AUC = 0.5: por cada punto de TPR se paga exactamente un punto de FPR. Aprobar/rechazar al azar dejaría pasar ≈50 % de los fraudes (pérdidas directas por contracargos) y a la vez bloquearía ≈50 % de las transacciones legítimas (clientes con pagos rechazados, ventas perdidas para los comercios, saturación del call center). Se combinan pérdidas financieras masivas, fuga de clientes, daño reputacional y posibles sanciones regulatorias. El modelo no aporta información: sería equivalente a no tener sistema de seguridad, pero con el costo de molestar a los clientes honestos.